In [5]:
import torch  
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

In [ ]:
X_train = [[1,14,22,5],[4,7,2,9,10,44],[3,2]]
y_train = [1,0,1]

max_length = 5
X_padded = np.zeros((len(X_train), max_length), dtype=int)
for i, review in enumerate(X_train):
    length = min(len(review), max_length)
    X_padded[i, :length] = review[:length]

X_tensor = torch.tensor(X_padded)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
class AdvancedLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(AdvancedLSTM, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, (hidden, cell) = self.lstm(embedded)

        final_hidden = hidden[-1, :, :]
        output = self.fc(final_hidden)

        return self.sigmoid(output).squeeze(1)

In [13]:
model = AdvancedLSTM(vocab_size=50, embedding_dim=10, hidden_dim=16, output_dim=1)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 3
for epoch in range(epochs):
    epoch_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()

        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)

        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

    # Missing block: Print progress per epoch to track training
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(dataloader):.4f}")

Epoch [1/3], Loss: 0.7070
Epoch [2/3], Loss: 0.7329
Epoch [3/3], Loss: 0.6997
